# NB-04-02 — First-Order Inference and Unification

**Module:** M04 · **Time:** 60–90 minutes · **Prerequisites:** L04-04, L04-05

You will unify terms, instantiate quantified rules, run forward and backward chaining over
the family and advising knowledge bases, and prove a query by resolution.

## 1. Setup

In [1]:
import sys
from pathlib import Path

for parent in [Path.cwd(), *Path.cwd().parents]:
    candidate = parent / "modules" / "M04-knowledge-representation-and-logical-reasoning" / "code"
    if candidate.is_dir():
        sys.path.insert(0, str(candidate))
        break
print("code directory:", candidate)

code directory: C:\Users\Nadeem\Documents\Artificial-Intelligence-Course\modules\M04-knowledge-representation-and-logical-reasoning\code


In [2]:
from fol_core import (
    Const, Var, func, pred, Forall, Exists, AndF, OrF, ImpliesF, NotF,
    unify, unify_atoms, substitute_formula, variables_in, plain,
    as_rule, forward_chain, backward_chain, fol_resolution,
)

## 2. Unification

In [3]:
x, y, z = Var("x"), Var("y"), Var("z")
parent = lambda a, b: pred("parent", a, b)
ancestor = lambda a, b: pred("ancestor", a, b)

print("x with bob:        ", unify(x, Const("bob")))
print("nested:            ", unify(func("f", x, Const("c")), func("f", Const("a"), y)))
print("shared variable:   ", unify(func("f", x, x), func("f", Const("a"), Const("b"))))
print("occurs check:      ", unify(x, func("f", x)))
print("predicate mismatch:", unify_atoms(parent(x, y), ancestor(x, y)))

x with bob:         {'x': Const(name='bob')}
nested:             {'x': Const(name='a'), 'y': Const(name='c')}
shared variable:    None
occurs check:       None
predicate mismatch: None


`None` means "no unifier". The shared-variable case fails because `x` would have to be
both `a` and `b`; the occurs check fails because `x = f(x)` has no finite solution.

## 3. Instantiating a quantified rule

In [4]:
rule = Forall(x, ImpliesF(pred("human", x), pred("mortal", x)))
print("rule:        ", plain(str(rule)))
print("body:        ", plain(str(rule.body)))
print("instantiated:", plain(str(substitute_formula(rule.body, {"x": Const("socrates")}))))
print("substituting in the whole formula:", plain(str(substitute_formula(rule, {"x": Const("socrates")}))))

rule:         forall x (human(x) -> mortal(x))
body:         (human(x) -> mortal(x))
instantiated: (human(socrates) -> mortal(socrates))
substituting in the whole formula: forall x (human(x) -> mortal(x))


The last line is unchanged: `x` is bound there, so substitution has nothing to do.
Universal instantiation drops the quantifier first, then substitutes.

## 4. Forward chaining over the family KB

In [5]:
from knowledge_bases import FAMILY, FAMILY_RULES, FAMILY_FACTS, ancestor, sibling

derived, log = forward_chain(FAMILY_RULES, FAMILY_FACTS)
for line in log:
    print(plain(line))
print("atoms in closure:", len(derived))

seed facts: 35
iter 1: ancestor(ann, bob) from rule 'a parent is an ancestor'
iter 1: ancestor(ann, cid) from rule 'a parent is an ancestor'
iter 1: ancestor(bob, dan) from rule 'a parent is an ancestor'
iter 1: ancestor(bob, eve) from rule 'a parent is an ancestor'
iter 1: ancestor(cid, finn) from rule 'a parent is an ancestor'
iter 1: sibling(bob, cid) from rule 'children of a common parent are siblings'
iter 1: sibling(cid, bob) from rule 'children of a common parent are siblings'
iter 1: sibling(dan, eve) from rule 'children of a common parent are siblings'
iter 1: sibling(eve, dan) from rule 'children of a common parent are siblings'
iter 2: ancestor(ann, dan) from rule 'an ancestor of a child is an ancestor'
iter 2: ancestor(ann, eve) from rule 'an ancestor of a child is an ancestor'
iter 2: ancestor(ann, finn) from rule 'an ancestor of a child is an ancestor'
fixpoint after 3 iteration(s)
atoms in closure: 47


Read the iteration numbers. The transitive rule needs a derived `ancestor` fact before
it can fire, which is why `ancestor(ann, finn)` appears in iteration 2 rather than 1.

## 5. Backward chaining, driven by the query

In [6]:
from knowledge_bases import DAN, CID

for solution in FAMILY.ask(ancestor(Var("who"), DAN)):
    print("ancestor of dan:", plain(str(solution["who"])))
print("siblings of cid:", [plain(str(s["who"])) for s in FAMILY.ask(sibling(CID, Var("who")))])

ancestor of dan: bob
ancestor of dan: ann
siblings of cid: ['bob']


## 6. The advising KB

In [7]:
from knowledge_bases import ADVISING, missing_prerequisites
from fol_core import Const

facts = ADVISING.close()
eligible = sorted(plain(str(a)) for a in facts if a.name == "eligible")
print("derived eligibility:")
for item in eligible:
    print("  ", item)
print()
print("chen -> cap400 missing:", missing_prerequisites("chen", "cap400"))
print("chen -> cap400 proof:  ", ADVISING.ask(pred("eligible", Const("chen"), Const("cap400"))))

derived eligibility:
   eligible(ali, cs201)
   eligible(ali, ds201)
   eligible(bea, cs201)
   eligible(bea, cs301)
   eligible(bea, ds201)
   eligible(chen, cs201)
   eligible(chen, cs301)
   eligible(chen, ds201)

chen -> cap400 missing: ['cs301']
chen -> cap400 proof:   []


The empty proof and the non-empty `missing` list are the same fact in two forms. The
logic says "not derivable"; the procedural check says *which prerequisite* is absent. That
is the closed-world assumption made explicit.

## 7. Resolution on a first-order KB

In [8]:
kb = [
    Forall(x, Forall(y, ImpliesF(pred("human", x), pred("mortal", x)))),
    pred("human", Const("socrates")),
]
proved, trace = fol_resolution(kb, pred("mortal", Const("socrates")))
print("proved:", proved)
for line in trace:
    print("  ", line)

proved: True
   start: 3 clauses
     given: mortal(x_0) or not human(x_0)
     given: human(socrates)
     given: not mortal(socrates)
   resolve mortal(x_0) or not human(x_0) with human(socrates) -> mortal(socrates)
   resolve mortal(x_0) or not human(x_0) with not mortal(socrates) -> not human(socrates)
   resolve human(socrates) with not human(socrates) -> empty clause


## 8. Exercise — extend the family

Add a `grandparent` rule and check both directions of inference agree.

In [9]:
grandparent = lambda a, b: pred("grandparent", a, b)
rule = as_rule(
    Forall(x, Forall(y, Forall(z, ImpliesF(
        AndF(parent(x, y), parent(y, z)), grandparent(x, z))))),
    "grandparent",
)
facts2, log2 = forward_chain(FAMILY_RULES + [rule], FAMILY_FACTS)
derived_gp = sorted(plain(str(a)) for a in facts2 if a.name == "grandparent")
print("forward:", derived_gp)
back = backward_chain(FAMILY_RULES + [rule], FAMILY_FACTS, grandparent(Var("who"), Const("finn")))
print("backward:", sorted(plain(str(s["who"])) for s in back))

forward: ['grandparent(ann, dan)', 'grandparent(ann, eve)', 'grandparent(ann, finn)']
backward: ['ann']


## 9. Reflect

1. Why must rules be standardised apart before matching?
2. Which direction would you choose for a system answering thousands of student queries?
3. What did the resolution trace show that the chaining trace did not?

Next: **LAB-04-01** (diagnostic rule engine) and **LAB-04-02** (advising and family).